# Data build internals

Internal reference for Stage 1 of the calibration pipeline: clone creation, source variable imputation, and PUF cloning.

**Requires:** `block_cd_distributions.csv.gz` in storage (from `make data`) for the clone creation cells. PUF cloning cells (Part 2) use toy DataFrames. Source imputation cells (Part 3) need ACS/SIPP/SCF donor files; those cells demonstrate the QRF concept with small synthetic data instead. 

---

## Pipeline overview

Stage 1 of the calibration pipeline transforms a raw CPS extract into an expanded dataset ready for the matrix-building step. The sequence is:

1. **PUF clone** — double the record count; impute 70+ tax variables on the second half from the PUF (`puf_impute.py` via `extended_cps.py`).
2. **Clone and assign geography** — replicate each record N times and give each clone a random census block drawn from a population-weighted distribution, with AGI-conditional routing for high-income households (`clone_and_assign.py`).
3. **Source imputation** — re-impute housing and asset variables from ACS, SIPP, and SCF donor surveys (`source_impute.py` via `create_source_imputed_cps.py`).

Because PUF cloning currently runs before geography assignment, the PUF QRF does not condition on state — both CPS and PUF halves receive geography only after doubling. `double_geography_for_puf()` exists for a planned future change where geography is assigned first, enabling state-conditional PUF imputation for richer geographic variation in tax variables.

See `calibration_package_internals.ipynb` for how these expanded records become columns in the calibration matrix.

---
## Part 1: Clone creation and geography assignment

### How cloning works

`assign_random_geography()` in `clone_and_assign.py` is the primary entry point. It does not literally duplicate rows in a DataFrame — it produces a `GeographyAssignment` object that indexes into the expanded record space. The caller is responsible for repeating the CPS arrays N times before writing them to the output dataset.

The function takes `n_records` (number of households in the base CPS) and `n_clones` and returns arrays of length `n_records * n_clones`. Index `i` in each array corresponds to clone `i // n_records`, record `i % n_records`.

### The `GeographyAssignment` dataclass

```python
@dataclass
class GeographyAssignment:
    block_geoid:  np.ndarray  # 15-char census block GEOIDs
    cd_geoid:     np.ndarray  # congressional district GEOIDs
    county_fips:  np.ndarray  # 5-char county FIPS codes
    state_fips:   np.ndarray  # 2-digit state FIPS integers
    n_records:    int
    n_clones:     int
```

- `block_geoid` is the primary key; `county_fips` is the first 5 characters, `state_fips` is the first 2 characters parsed as an integer.
- `cd_geoid` uses the format `state_fips * 100 + district_number`. At-large districts (district 00) and DC (district 98) are normalized to district 01.

### The no-collision constraint

The algorithm samples blocks independently per clone, but enforces that **the same CPS record cannot land in the same congressional district in two different clones**. Without this constraint, a high-weight record in a small district could dominate the calibration target for that district across multiple clones.

Implementation:
- Clone 0 draws freely.
- Each subsequent clone checks for collisions against all previous clones and resamples the colliding records, up to 50 retries.
- Residual collisions after 50 retries are accepted (very rare with large block distributions).

### AGI-conditional geographic assignment

When `household_agi` and `cd_agi_targets` are provided, `assign_random_geography()` uses a two-distribution sampling strategy:

1. **Identify extreme households** — those at or above the `agi_threshold_pctile` (default 90th percentile) of household AGI.
2. **Build AGI-weighted block probabilities** — `_build_agi_block_probs()` multiplies population block probabilities by CD-level AGI targets: `P_agi(block) = P_pop(block) * AGI_target(CD) / Z`. This makes blocks in high-AGI districts more likely for extreme households.
3. **Split sampling** — extreme households draw from `P_agi`; all other households draw from the standard population-weighted `P_pop`.
4. **Collision resampling respects the split** — when retrying collisions, extreme households resample from `P_agi` and normal households from `P_pop`.

**Why this matters:** Without AGI-conditional assignment, a high-AGI household could land in a low-AGI district by chance. The L0 optimizer would then zero that record's weight to match the district's low AGI target — destroying population targets in the process. By routing high-income households toward high-AGI districts, the initial placement is more compatible with what calibration needs, and the optimizer can retain these records without sacrificing other targets.

The CD-level AGI targets are loaded from `policy_data.db` in `run_calibration()` (in `unified_calibration.py`), which queries the `targets` table for active district-level `adjusted_gross_income` targets.

### Geography is rederived, not persisted

The `GeographyAssignment` is held in memory and passed through function calls during a pipeline run. It is **not** serialized to disk — each worker process calls `assign_random_geography()` with the same deterministic seed, so reproducibility is guaranteed without saving state. This avoids stale `.npz` files drifting out of sync with the block distribution data.

### Demonstration: geography assignment on 20 households, 3 clones

The real `assign_random_geography()` requires `block_cd_distributions.csv.gz` in the storage folder. Here we call it directly (it will load the distribution from storage). The call below also shows the AGI-conditional parameters; we pass `None` for both since CD AGI targets require a populated `policy_data.db`, but the function signature is demonstrated.

In [ ]:
import numpy as np
import pandas as pd
from policyengine_us_data.calibration.clone_and_assign import (
    GeographyAssignment,
    assign_random_geography,
    double_geography_for_puf,
)

N_RECORDS = 20  # 20 base CPS households
N_CLONES = 3  # 3 geographic replicas
# In production: N_RECORDS ~12,000, N_CLONES=430 -> ~5.2M matrix columns.

# In production, run_calibration() computes household AGI and loads
# CD AGI targets from policy_data.db, then passes them here:
#   household_agi=base_agi,          # np.ndarray of per-household AGI
#   cd_agi_targets=cd_agi_targets,   # dict mapping CD GEOID str -> AGI $
# Here we pass None for both (falls back to uniform population-weighted sampling).

geo = assign_random_geography(
    n_records=N_RECORDS,
    n_clones=N_CLONES,
    seed=42,
    household_agi=None,
    cd_agi_targets=None,
)

print(f"n_records={geo.n_records}, n_clones={geo.n_clones}")
print(f"Total column positions: {geo.n_records * geo.n_clones}")
print(f"\nFirst 4 column positions:")
for i in range(4):
    print(
        f"  col {i}: block={geo.block_geoid[i]}  cd={geo.cd_geoid[i]}  state={geo.state_fips[i]}"
    )

In [2]:
# Show the assignment as a table: rows = records, columns = clones
rows = []
for rec in range(N_RECORDS):
    row = {"record": rec}
    for clone in range(N_CLONES):
        flat_idx = clone * N_RECORDS + rec
        row[f"clone_{clone}_cd"] = geo.cd_geoid[flat_idx]
        row[f"clone_{clone}_state"] = geo.state_fips[flat_idx]
    rows.append(row)

df_geo = pd.DataFrame(rows)
print(df_geo.to_string(index=False))

# Verify the no-collision property: no record shares the same CD across two clones
cd_matrix = geo.cd_geoid.reshape(N_CLONES, N_RECORDS)
collision_check = np.zeros(N_RECORDS, dtype=bool)
for c1 in range(N_CLONES):
    for c2 in range(c1 + 1, N_CLONES):
        collision_check |= cd_matrix[c1] == cd_matrix[c2]
print(
    f"\nRecords with same CD in any two clones: {collision_check.sum()} (should be 0)"
)

 record clone_0_cd  clone_0_state clone_1_cd  clone_1_state clone_2_cd  clone_2_state
      0       4213             42       4206             42       2501             25
      1       2502             25       1804             18       4803             48
      2       4814             48       5309             53       3907             39
      3       3906             39       4829             48       1703             17
      4        621              6       4215             42       4802             48
      5       5401             54        904              9       4507             45
      6       4207             42       2605             26       2101             21
      7       4401             44        503              5       1310             13
      8        635              6        647              6       3714             37
      9       2507             25       3714             37        640              6
     10       1902             19       4106          

### `double_geography_for_puf()`

After `puf_clone_dataset()` doubles the record count (one CPS half, one PUF half), the geometry must also double. `double_geography_for_puf()` iterates over each clone's slice and concatenates it with itself:

```
Clone c before PUF doubling:  [rec_0, rec_1, ..., rec_{n-1}]
Clone c after PUF doubling:   [rec_0, ..., rec_{n-1}, rec_0, ..., rec_{n-1}]
                                \________CPS half________/ \______PUF half______/
```

The CPS half and its PUF copy share the identical geographic assignment. `n_records` doubles from `n` to `2n`; `n_clones` stays the same.

In [4]:
geo_doubled = double_geography_for_puf(geo)

print(f"Before doubling: n_records={geo.n_records}")
print(f"After doubling:  n_records={geo_doubled.n_records}")
print(f"Total positions before: {geo.n_records * geo.n_clones}")
print(f"Total positions after:  {geo_doubled.n_records * geo_doubled.n_clones}")

# Invariant: for every clone, CPS half and PUF half share the same geography.
for c in range(N_CLONES):
    start = c * geo_doubled.n_records
    cps_cds = geo_doubled.cd_geoid[start : start + N_RECORDS]
    puf_cds = geo_doubled.cd_geoid[start + N_RECORDS : start + geo_doubled.n_records]
    assert np.array_equal(cps_cds, puf_cds), f"Clone {c}: CPS/PUF geography mismatch"
print("\nCPS and PUF halves share identical geography in all clones.")

Before doubling: n_records=20
After doubling:  n_records=40
Total positions before: 60
Total positions after:  120

CPS and PUF halves share identical geography in all clones.


---
## Part 2: PUF cloning and tax variable imputation*

*Currently happens before geography assignment.

`puf_clone_dataset()` in `puf_impute.py` doubles the record count. After this step the dataset has `2 * n_records` total person records: one CPS half and one PUF half. This runs before geography assignment, so the PUF QRF currently conditions only on demographics and income — not on state.

### The 2-clone structure

| Half | Records | Key properties |
|------|---------|----------------|
| CPS half | indices `0 .. n_cps - 1` | Retains original CPS values for most variables; `OVERRIDDEN_IMPUTED_VARIABLES` are re-imputed from PUF for both halves |
| PUF half | indices `n_cps .. 2*n_cps - 1` | All `IMPUTED_VARIABLES` replaced with PUF QRF predictions; **all weight arrays zeroed** |

The logic inside `puf_clone_dataset()` selects the treatment for each variable:

```python
if variable in OVERRIDDEN_IMPUTED_VARIABLES:
    # Both halves get PUF predictions
    new_data[variable] = concatenate([pred, pred])
elif variable in IMPUTED_VARIABLES:
    # CPS half keeps original; PUF half gets predictions
    new_data[variable] = concatenate([values, pred])
elif "_id" in variable:
    # IDs must be unique: PUF IDs are offset by max(CPS IDs)
    new_data[variable] = concatenate([values, values + values.max()])
elif "_weight" in variable:
    # PUF half starts with zero weight
    new_data[variable] = concatenate([values, values * 0])
else:
    # Default: duplicate unchanged
    new_data[variable] = concatenate([values, values])
```

### Why the PUF half starts with zero weight

The calibration matrix optimizer assigns weights to every record across all clones. At initialization, giving the PUF half zero weight means it contributes nothing to population totals until the optimizer decides it should. This prevents PUF records — which are imputed rather than survey-observed — from biasing the initial distribution before calibration has had a chance to learn from targets.

The zeroing is a single line: `values * 0` applied to any variable whose name contains `"_weight"`. After calibration the PUF records receive positive weights wherever they improve the fit to tax-variable targets (e.g., total capital gains, interest income).

In [6]:
import numpy as np
import pandas as pd

# Toy 2-clone structure: 5 households, a subset of variables
n_cps = 5
time_period = 2024

toy_data = {
    "household_id": {time_period: np.array([101, 102, 103, 104, 105])},
    "person_id": {time_period: np.array([201, 202, 203, 204, 205])},
    "household_weight": {time_period: np.array([1.2, 3.4, 2.1, 0.8, 1.9])},
    "employment_income": {
        time_period: np.array([45_000, 0, 80_000, 30_000, 60_000], dtype=np.float32)
    },
    "social_security": {
        time_period: np.array([0, 18_000, 0, 12_000, 0], dtype=np.float32)
    },
    "age": {time_period: np.array([38, 72, 45, 68, 52], dtype=np.float32)},
}

# Simulate PUF QRF predictions for IMPUTED_VARIABLES (here just employment_income)
puf_employment_preds = np.array(
    [52_000, 1_000, 90_000, 28_000, 71_000], dtype=np.float32
)

IMPUTED_VARIABLES_TOY = {"employment_income"}

new_data = {}
for variable, time_dict in toy_data.items():
    values = time_dict[time_period]
    if variable in IMPUTED_VARIABLES_TOY:
        pred = puf_employment_preds
        new_data[variable] = {time_period: np.concatenate([values, pred])}
    elif "_id" in variable:
        new_data[variable] = {
            time_period: np.concatenate([values, values + values.max()])
        }
    elif "_weight" in variable:
        new_data[variable] = {time_period: np.concatenate([values, values * 0])}
    else:
        new_data[variable] = {time_period: np.concatenate([values, values])}

summary = pd.DataFrame(
    {
        "half": ["CPS"] * n_cps + ["PUF"] * n_cps,
        "household_id": new_data["household_id"][time_period],
        "household_weight": new_data["household_weight"][time_period],
        "employment_income": new_data["employment_income"][time_period],
        "social_security": new_data["social_security"][time_period],
        "age": new_data["age"][time_period],
    }
)
print(summary.to_string(index=False))

half  household_id  household_weight  employment_income  social_security  age
 CPS           101               1.2            45000.0              0.0 38.0
 CPS           102               3.4                0.0          18000.0 72.0
 CPS           103               2.1            80000.0              0.0 45.0
 CPS           104               0.8            30000.0          12000.0 68.0
 CPS           105               1.9            60000.0              0.0 52.0
 PUF           206               0.0            52000.0              0.0 38.0
 PUF           207               0.0             1000.0          18000.0 72.0
 PUF           208               0.0            90000.0              0.0 45.0
 PUF           209               0.0            28000.0          12000.0 68.0
 PUF           210               0.0            71000.0              0.0 52.0


The CPS half retains its original `employment_income`; the PUF half receives QRF-imputed values. Household IDs are offset in the PUF half to prevent collisions (`household_id` goes from 101–105 to 106–110). Weights are 0.0 across the entire PUF half.

### The 70+ tax variables imputed from PUF

`IMPUTED_VARIABLES` contains 70 variables covering the main tax return line items. `OVERRIDDEN_IMPUTED_VARIABLES` (a subset of 44 variables) are imputed for **both** CPS and PUF halves — these are variables where the CPS estimate is considered unreliable enough that the PUF imputation is preferred even for records that stay in the CPS half.

A representative selection from `IMPUTED_VARIABLES`:
- Income: `employment_income`, `self_employment_income`, `partnership_s_corp_income`, `rental_income`, `farm_income`, `estate_income`, `alimony_income`
- Capital: `long_term_capital_gains`, `short_term_capital_gains`, `qualified_dividend_income`, `non_qualified_dividend_income`, `taxable_interest_income`, `tax_exempt_interest_income`
- Deductions: `charitable_cash_donations`, `charitable_non_cash_donations`, `deductible_mortgage_interest`, `student_loan_interest`, `health_savings_account_ald`
- Social Security: `social_security` (total; sub-components reconciled separately)
- Retirement: `taxable_ira_distributions`, `taxable_pension_income`, `tax_exempt_pension_income`
- Credits: `foreign_tax_credit`, `american_opportunity_credit`, `savers_credit`, `general_business_credit`
- Qualified business income: `w2_wages_from_qualified_business`, `unadjusted_basis_qualified_property`, `qualified_reit_and_ptp_income`, `qualified_bdc_income`

### Stratified subsampling of PUF training data

`_stratified_subsample_index()` in `_run_qrf_imputation()` selects training records from the PUF:
- Keeps **all** records with AGI at or above the 99.5th percentile (top 0.5%, `PUF_TOP_PERCENTILE = 99.5`)
- Randomly samples the remainder to reach a target of 20,000 total (`PUF_SUBSAMPLE_TARGET = 20_000`)

This preserves the extreme tail of the AGI distribution in the training set, which matters for rare but high-impact income types (capital gains, partnership income).

### Sequential QRF: preserving the joint distribution

`_sequential_qrf()` calls `microimpute.QRF.fit_predict()`, which imputes each variable **conditioned on all previously imputed variables** in the `output_vars` list order. This preserves the joint distribution across the 70 tax variables — for example, records with high `employment_income` will tend to receive plausible values for `pre_tax_contributions` because the model sees both.

In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Minimal illustration of sequential conditioning:
# Impute capital_gains conditioned on employment_income,
# then impute dividends conditioned on both.

rng = np.random.default_rng(1)

# PUF-like donor: employment correlated with capital gains; gains correlated with dividends
n_donor = 300
emp = rng.exponential(50_000, n_donor)
age = rng.integers(25, 80, n_donor).astype(float)
cg = np.maximum(0, 0.05 * emp + rng.normal(0, 5_000, n_donor))
div = np.maximum(0, 0.3 * cg + rng.normal(0, 500, n_donor))

donor = pd.DataFrame(
    {"age": age, "employment_income": emp, "capital_gains": cg, "dividends": div}
)

# CPS receiver: only demographics known initially
n_recv = 6
recv = pd.DataFrame(
    {
        "age": [30, 45, 60, 35, 55, 70],
        "employment_income": [40_000, 80_000, 20_000, 55_000, 95_000, 5_000],
    }
)

# Step 1: impute capital_gains from demographics alone
rf1 = RandomForestRegressor(n_estimators=50, random_state=0)
rf1.fit(donor[["age", "employment_income"]], donor["capital_gains"])
recv["capital_gains"] = rf1.predict(recv[["age", "employment_income"]]).round(0)

# Step 2: impute dividends conditioning on capital_gains (now available)
rf2 = RandomForestRegressor(n_estimators=50, random_state=0)
rf2.fit(donor[["age", "employment_income", "capital_gains"]], donor["dividends"])
recv["dividends"] = rf2.predict(
    recv[["age", "employment_income", "capital_gains"]]
).round(0)

print(recv.to_string(index=False))
print(
    f"\nDonor corr(capital_gains, dividends): {donor['capital_gains'].corr(donor['dividends']):.3f}"
)
print(
    f"Imputed corr(capital_gains, dividends): {recv['capital_gains'].corr(recv['dividends']):.3f}"
)

 age  employment_income  capital_gains  dividends
  30              40000         3143.0     1168.0
  45              80000         3564.0      767.0
  60              20000         1584.0      770.0
  35              55000         1713.0      488.0
  55              95000         2673.0      881.0
  70               5000         1497.0     1035.0

Donor corr(capital_gains, dividends): 0.941
Imputed corr(capital_gains, dividends): 0.281


By conditioning each variable on those already imputed, the sequential QRF preserves inter-variable correlations from the donor distribution.

### Social Security sub-component reconciliation

The PUF records a total `social_security` amount but not its breakdown by reason code. `reconcile_ss_subcomponents()` derives the four sub-components for every PUF record with positive SS:

```
SS_SUBCOMPONENTS = [
    "social_security_retirement",
    "social_security_disability",
    "social_security_survivors",
    "social_security_dependents",
]
```

**Step 1 — QRF share prediction:** `_qrf_ss_shares()` trains on CPS records where all four sub-components are known. It converts each sub-component to a share of the total (`sub_value / ss_total`), trains a QRF on these shares using `SS_SPLIT_PREDICTORS` (`age`, `is_male`, `tax_unit_is_joint`, `is_tax_unit_head`, `is_tax_unit_dependent`), and predicts shares for PUF records. Predicted shares are clipped to [0, 1] and renormalized to sum to 1.

**Step 2 — Age heuristic fallback:** If the QRF fails (fewer than 100 CPS training records or a runtime exception), `_age_heuristic_ss_shares()` assigns:
- Age ≥ 62 (`MINIMUM_RETIREMENT_AGE`): 100% retirement
- Age < 62: 100% disability
- If age is unavailable: 100% retirement for all

**Step 3 — Scale to imputed total:** Each sub-component is set to `ss_total * share`. PUF records with `social_security == 0` get all sub-components set to 0.

The function modifies `data` in place, only touching the PUF half (indices `n_cps .. 2*n_cps`).

In [8]:
import numpy as np
import pandas as pd

# Toy SS reconciliation: 4 PUF records with positive SS
# Predicted shares come from the QRF (here we supply them directly).

puf_ss_total = np.array([24_000, 14_400, 9_600, 18_000])  # imputed totals
ages = np.array([70, 55, 68, 45])  # ages for heuristic

# Suppose QRF predicted these normalized shares for each record
qrf_shares = {
    "social_security_retirement": np.array([0.90, 0.00, 0.85, 0.00]),
    "social_security_disability": np.array([0.05, 0.95, 0.10, 0.90]),
    "social_security_survivors": np.array([0.03, 0.03, 0.03, 0.05]),
    "social_security_dependents": np.array([0.02, 0.02, 0.02, 0.05]),
}

# Renormalize (shares already sum to ~1 here; shown for completeness)
total_shares = sum(qrf_shares.values())
for k in qrf_shares:
    qrf_shares[k] = np.where(total_shares > 0, qrf_shares[k] / total_shares, 0.0)

# Scale to imputed total
result = pd.DataFrame({"age": ages, "ss_total": puf_ss_total})
for sub, shares in qrf_shares.items():
    result[sub.replace("social_security_", "ss_")] = (puf_ss_total * shares).round(0)

result["check_sum"] = (
    result["ss_retirement"]
    + result["ss_disability"]
    + result["ss_survivors"]
    + result["ss_dependents"]
)
print(result.to_string(index=False))
print(
    "\nAll check sums match ss_total:",
    np.allclose(result["check_sum"], result["ss_total"], atol=1),
)

 age  ss_total  ss_retirement  ss_disability  ss_survivors  ss_dependents  check_sum
  70     24000        21600.0         1200.0         720.0          480.0    24000.0
  55     14400            0.0        13680.0         432.0          288.0    14400.0
  68      9600         8160.0          960.0         288.0          192.0     9600.0
  45     18000            0.0        16200.0         900.0          900.0    18000.0

All check sums match ss_total: True


### Age heuristic fallback: numeric example

In [9]:
MINIMUM_RETIREMENT_AGE = 62

ages = np.array([70, 55, 68, 45])
puf_ss_total = np.array([24_000, 14_400, 9_600, 18_000])

is_old = ages >= MINIMUM_RETIREMENT_AGE

heuristic_shares = {
    "social_security_retirement": is_old.astype(float),
    "social_security_disability": (~is_old).astype(float),
    "social_security_survivors": np.zeros(len(ages)),
    "social_security_dependents": np.zeros(len(ages)),
}

heuristic_result = pd.DataFrame(
    {"age": ages, "ss_total": puf_ss_total, "is_retirement_age": is_old}
)
for sub, shares in heuristic_shares.items():
    heuristic_result[sub.replace("social_security_", "ss_")] = (
        puf_ss_total * shares
    ).round(0)

print(heuristic_result.to_string(index=False))

 age  ss_total  is_retirement_age  ss_retirement  ss_disability  ss_survivors  ss_dependents
  70     24000               True        24000.0            0.0           0.0            0.0
  55     14400              False            0.0        14400.0           0.0            0.0
  68      9600               True         9600.0            0.0           0.0            0.0
  45     18000              False            0.0        18000.0           0.0            0.0


When the QRF heuristic applies, the two youngest recipients (ages 55 and 45) have their entire SS imputed as disability; the two older recipients get it all as retirement. Survivors and dependents receive nothing under the heuristic.

### Additional PUF-half imputations

Beyond the 70 IMPUTED_VARIABLES, two further QRF passes run on the PUF half:

**`_impute_weeks_unemployed()`** — Trains on CPS records where `weeks_unemployed` is observed. The test predictor set uses PUF-imputed `taxable_unemployment_compensation` as an additional feature (when available), ensuring the imputed weeks are consistent with the PUF unemployment income values. Predictions are clipped to [0, 52]; records with zero unemployment compensation receive zero weeks.

**`_impute_retirement_contributions()`** — Imputes `traditional_401k_contributions`, `roth_401k_contributions`, `traditional_ira_contributions`, `roth_ira_contributions`, and `self_employed_pension_contributions` for the PUF half. The test data uses CPS demographic predictors (`RETIREMENT_DEMOGRAPHIC_PREDICTORS`) combined with PUF-imputed income (`RETIREMENT_INCOME_PREDICTORS`). After prediction, year-specific IRS contribution limits are applied:
- 401k: capped at the annual limit + catch-up allowance for age ≥ 50
- IRA: capped at the annual IRA limit + catch-up for age ≥ 50
- SE pension: capped at `min(25% of SE income, dollar limit)` from `imputation_parameters.yaml`
- All contributions zeroed for records without the corresponding income type

### End-to-end record count accounting

In [10]:
import pandas as pd

n_base_cps = 100_000  # typical household count in base CPS
n_clones = 10

stages = [
    ("Base CPS", n_base_cps, "-"),
    ("After clone x10", n_base_cps * n_clones, "assign_random_geography()"),
    ("After PUF doubling", n_base_cps * n_clones * 2, "puf_clone_dataset()"),
    ("After double_geography", n_base_cps * n_clones * 2, "double_geography_for_puf()"),
]

df_stages = pd.DataFrame(
    stages, columns=["Stage", "Household records", "Primary function"]
)
print(df_stages.to_string(index=False))
print(f"\nFinal n_records in GeographyAssignment: {n_base_cps * 2}")
print(f"Final n_clones in GeographyAssignment:  {n_clones}")
print(f"Total flat length of each geo array:    {n_base_cps * 2 * n_clones:,}")

                 Stage  Household records           Primary function
              Base CPS             100000                          -
       After clone x10            1000000  assign_random_geography()
    After PUF doubling            2000000        puf_clone_dataset()
After double_geography            2000000 double_geography_for_puf()

Final n_records in GeographyAssignment: 200000
Final n_clones in GeographyAssignment:  10
Total flat length of each geo array:    2,000,000


### Summary of variable treatment in `puf_clone_dataset()`

| Variable class | Examples | CPS half value | PUF half value |
|----------------|----------|---------------|----------------|
| `IMPUTED_VARIABLES` | `employment_income`, `social_security`, capital gains | Original CPS value | QRF prediction from PUF |
| `OVERRIDDEN_IMPUTED_VARIABLES` | `partnership_s_corp_income`, `deductible_mortgage_interest` | PUF QRF prediction | PUF QRF prediction |
| ID variables (`*_id`) | `household_id`, `person_id` | Original | Offset by `max(CPS IDs)` |
| Weight variables (`*_weight`) | `household_weight` | Original | **0.0** |
| `SS_SUBCOMPONENTS` | `social_security_retirement` | Original CPS split | Reconciled from imputed total |
| `CPS_RETIREMENT_VARIABLES` | `traditional_401k_contributions` | Original CPS value | QRF from CPS trained on retirement income |
| `weeks_unemployed` | — | Original CPS value | QRF conditioned on PUF unemployment comp |
| All other variables | `age`, `rent`, `state_fips` | Original | Duplicated unchanged |

---

See `calibration_package_internals.ipynb` for how these expanded, imputed records are assembled into the calibration matrix and how weights are optimized across all clones.

---
## Part 3: Source variable imputation (ACS/SIPP/SCF)

After PUF cloning and geography assignment, `impute_source_variables()` in `source_impute.py` re-imputes 9 variables that the CPS measures poorly. All three donor surveys use Quantile Random Forest (QRF) from the `microimpute` package. Because geography is already assigned at this point, the ACS QRF can condition on `state_fips`.

### Variables and donor surveys

| Survey | Variables imputed | State predictor? |
|--------|------------------|------------------|
| ACS 2022 | `rent`, `real_estate_taxes` | **Yes** (`state_fips`) |
| SIPP 2023 | `tip_income`, `bank_account_assets`, `stock_assets`, `bond_assets` | No |
| SCF 2022 | `net_worth`, `auto_loan_balance`, `auto_loan_interest` | No |

### Why only ACS includes a state predictor

The ACS is a large household survey with published state identifiers, so the donor file includes `state_fips` for every record. The QRF trained on ACS data therefore learns state-level differences in rent levels and property tax burdens, and propagates those differences to the CPS clones (which now carry `state_fips` from the geography assignment step).

SIPP and SCF do not publish state identifiers in their public-use files. Their QRF imputations condition only on demographic and financial predictors, so the imputed values vary across individuals but not across states. This is a deliberate simplification: the pipeline accepts state-blind asset imputations because assets are less geographically concentrated than housing costs.

### Predictor sets

**ACS predictors** (+ `state_fips`):
`is_household_head`, `age`, `is_male`, `tenure_type`, `employment_income`, `self_employment_income`, `social_security`, `pension_income`, `household_size`

**SIPP tip predictors:**
`employment_income`, `age`, `count_under_18`, `count_under_6`

**SIPP asset predictors:**
`employment_income`, `age`, `is_female`, `is_married`, `count_under_18`

**SCF predictors:**
`age`, `is_female`, `cps_race`, `is_married`, `own_children_in_household`, `employment_income`, `interest_dividend_income`, `social_security_pension_income`

### Training sample sizes

- ACS: 10,000 household heads sampled from ACS 2022
- SIPP tips: up to 10,000 records sampled with probability proportional to `WPFINWGT`
- SIPP assets: up to 20,000 records sampled with probability proportional to `WPFINWGT`
- SCF: 50% random sample of SCF 2022

### Toy QRF imputation: state-aware vs state-blind

The following cell builds a minimal synthetic example showing how including `state_fips` in the predictor set causes the QRF to produce state-differentiated predictions, while omitting it collapses predictions to demographic-only variation.

In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(0)

# Synthetic donor (ACS-like): 200 records across 4 states.
# Rent in CA (state 6) is structurally higher than in AL (state 1).
n_donor = 200
states_donor = rng.choice([1, 6, 12, 36], size=n_donor)  # AL, CA, FL, NY
age_donor = rng.integers(25, 75, size=n_donor).astype(float)
income_donor = rng.exponential(40_000, size=n_donor)
state_premium = np.where(
    states_donor == 6,
    1200,  # CA
    np.where(
        states_donor == 36,
        900,  # NY
        np.where(
            states_donor == 12,
            400,  # FL
            200,
        ),
    ),
)  # AL
rent_donor = state_premium + 0.005 * income_donor + rng.normal(0, 100, n_donor)
rent_donor = np.clip(rent_donor, 0, None)

donor_df = pd.DataFrame(
    {
        "age": age_donor,
        "income": income_donor,
        "state_fips": states_donor.astype(float),
        "rent": rent_donor,
    }
)

# Synthetic receiver: 12 CPS records, 4 per state
receiver_df = pd.DataFrame(
    {
        "age": [35.0, 45.0, 55.0, 30.0] * 3,
        "income": [30_000, 60_000, 80_000, 25_000] * 3,
        "state_fips": [1.0] * 4 + [6.0] * 4 + [36.0] * 4,
        "state_label": ["AL"] * 4 + ["CA"] * 4 + ["NY"] * 4,
    }
)

# State-AWARE model
rf_aware = RandomForestRegressor(n_estimators=100, random_state=0)
rf_aware.fit(donor_df[["age", "income", "state_fips"]], donor_df["rent"])
receiver_df["pred_state_aware"] = rf_aware.predict(
    receiver_df[["age", "income", "state_fips"]]
).round(0)

# State-BLIND model (same demographics, no state_fips)
rf_blind = RandomForestRegressor(n_estimators=100, random_state=0)
rf_blind.fit(donor_df[["age", "income"]], donor_df["rent"])
receiver_df["pred_state_blind"] = rf_blind.predict(
    receiver_df[["age", "income"]]
).round(0)

print(
    receiver_df[
        ["state_label", "age", "income", "pred_state_aware", "pred_state_blind"]
    ].to_string(index=False)
)

state_label  age  income  pred_state_aware  pred_state_blind
         AL 35.0   30000             282.0            1151.0
         AL 45.0   60000             599.0            1200.0
         AL 55.0   80000             590.0             692.0
         AL 30.0   25000             259.0             905.0
         CA 35.0   30000            1358.0            1151.0
         CA 45.0   60000            1618.0            1200.0
         CA 55.0   80000            1524.0             692.0
         CA 30.0   25000            1213.0             905.0
         NY 35.0   30000            1112.0            1151.0
         NY 45.0   60000            1199.0            1200.0
         NY 55.0   80000            1270.0             692.0
         NY 30.0   25000            1147.0             905.0


The state-aware predictions separate AL from CA and NY households with the same age and income. The state-blind predictions are identical for those records — any geographic variation in rent then depends entirely on which clones happen to overlap with which calibration targets, rather than on the underlying rent distribution.

---

After Part 3, the base datasets are ready. Sometimes these are used by themselves, for microsimulation use or for other branches of the data pipeline. However, their biggest use is as a base for national and local area calibration. See `calibration_package_internals.ipynb` for an explanation of how the calibration problem is set up to build on them.